# Supervised Fine-tuning (SFT) with Unsloth

Fine-tuning requires a GPU. If you don't have one locally, you can run this notebook for free on [Google Colab](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/finetuning/notebooks/sft_with_unsloth.ipynb) using a free NVIDIA T4 GPU instance.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/finetuning/notebooks/sft_with_unsloth.ipynb)

### What's in this notebook?

In this notebook you will learn how to perform Supervised Fine-tuning (SFT) using Unsloth for efficient, memory-optimized training.
We will use the [LFM2.5-1.2B-Instruct](https://docs.liquid.ai/docs/models/lfm25-1.2b-instruct) model and fine-tune it on the FineTome-100k dataset using LoRA adapters and Unsloth's optimizations for 2x faster training with reduced memory consumption.

We will cover
- Environment setup
- Data preparation
- Model training
- Local inference with your new model
- Model saving and exporting it into the format you need for **deployment**.

### Deployment options

LFM2.5 models are small and efficient, enabling deployment across a wide range of platforms:

<table align="left">
  <tr>
    <th>Deployment Target</th>
    <th>Use Case</th>
  </tr>
  <tr>
    <td>📱 <a href="https://docs.liquid.ai/leap/edge-sdk/android/android-quick-start-guide"><b>Android</b></a></td>
    <td>Mobile apps on Android devices</td>
  </tr>
  <tr>
    <td>📱 <a href="https://docs.liquid.ai/leap/edge-sdk/ios/ios-quick-start-guide"><b>iOS</b></a></td>
    <td>Mobile apps on iPhone/iPad</td>
  </tr>
  <tr>
    <td>🍎 <a href="https://docs.liquid.ai/docs/inference/mlx"><b>Apple Silicon Mac</b></a></td>
    <td>Local inference on Mac with MLX</td>
  </tr>
  <tr>
    <td>🦙 <a href="https://docs.liquid.ai/docs/inference/llama-cpp"><b>llama.cpp</b></a></td>
    <td>Local deployments on any hardware</td>
  </tr>
  <tr>
    <td>🦙 <a href="https://docs.liquid.ai/docs/inference/ollama"><b>Ollama</b></a></td>
    <td>Local inference with easy setup</td>
  </tr>
  <tr>
    <td>🖥️ <a href="https://docs.liquid.ai/docs/inference/lm-studio"><b>LM Studio</b></a></td>
    <td>Desktop app for local inference</td>
  </tr>
  <tr>
    <td>⚡ <a href="https://docs.liquid.ai/docs/inference/vllm"><b>vLLM</b></a></td>
    <td>Cloud deployments with high throughput</td>
  </tr>
  <tr>
    <td>☁️ <a href="https://docs.liquid.ai/docs/inference/modal-deployment"><b>Modal</b></a></td>
    <td>Serverless cloud deployment</td>
  </tr>
  <tr>
    <td>🏗️ <a href="https://docs.liquid.ai/docs/inference/baseten-deployment"><b>Baseten</b></a></td>
    <td>Production ML infrastructure</td>
  </tr>
  <tr>
    <td>🚀 <a href="https://docs.liquid.ai/docs/inference/fal-deployment"><b>Fal</b></a></td>
    <td>Fast inference API</td>
  </tr>
</table>

### Need help building with our models and tools?
Join the Liquid AI Discord Community and ask.

<a href="https://discord.com/invite/liquid-ai"><img src="https://img.shields.io/discord/1385439864920739850?color=7289da&label=Join%20Discord&logo=discord&logoColor=white" alt="Join Discord"></a>

And now, let the fine tune begin!

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.3
!pip install --no-deps trl==0.22.2

### Unsloth

In [1]:
from unsloth import FastLanguageModel
import torch

print("Loading LFM2.5-1.2B-Instruct from LiquidAI...")
model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name="unsloth/LFM2.5-1.2B-Instruct",
    model_name="LiquidAI/LFM2.5-1.2B-Instruct",
    max_seq_length = 16000, # Choose any for long context!
    load_in_4bit = False, # 4 bit quantization to reduce memory
    load_in_8bit = True, # [NEW!] A bit more accurate, uses 2x memory
    load_in_16bit = False, # [NEW!] Enables 16bit LoRA
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)
print("Model loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


D:\Cache\conda\envs\finetuner\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0314 04:12:12.668000 21316 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
Loading LFM2.5-1.2B-Instruct from LiquidAI...
==((====))==  Unsloth 2026.3.4: Fast Lfm2 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 3050 6GB Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|█████| 148/148 [00:09<00:00, 16.34it/s, Materializing param=model.layers.15.operator_norm.weight]


Model loaded successfully!


## Add LoRA Adapters

We now add LoRA adapters so we only need to update a small amount of parameters!

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "out_proj", "in_proj",
                      "w1", "w2", "w3"],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Making `model.base_model.model.model` require gradients


<a name="Data"></a>
### Data Prep
We now use the `LFM` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. LFM renders multi turn conversations like below:

```
<|startoftext|><|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>
```

In [3]:
tokenizer.apply_chat_template([
    {"role" : "user", "content" : "Hello, I am Parampreet Singh."},
    {"role" : "assistant", "content" : "Hey there, Param!"}
], tokenize = False)

'<|startoftext|><|im_start|>user\nHello, I am Parampreet Singh.<|im_end|>\n<|im_start|>assistant\nHey there, Param!<|im_end|>\n'

We get the first ~3000~ 1000 rows of the dataset

In [4]:
from datasets import load_dataset
dataset = load_dataset("mlabonne/FineTome-100k", split = "train[:1000]")

We now use `standardize_data_formats` to try converting datasets to the correct format for finetuning purposes!

In [5]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

Let's see how row 100 looks like!

In [6]:
sample = dataset[37]
print(sample.keys())
for k, v in sample.items():
    print(f"{k} : {v}")
print("="*50)
for i, conv in enumerate(sample["conversations"]):
  print(f"Conversation {i}")
  for k, v in conv.items():
    print(f"{k} : {v[:50]}")

dict_keys(['conversations', 'source', 'score'])
conversations : [{'content': 'Write Python code to solve the task:\nIn programming, hexadecimal notation is often used.\n\nIn hexadecimal notation, besides the ten digits 0, 1, ..., 9, the six letters `A`, `B`, `C`, `D`, `E` and `F` are used to represent the values 10, 11, 12, 13, 14 and 15, respectively.\n\nIn this problem, you are given two letters X and Y. Each X and Y is `A`, `B`, `C`, `D`, `E` or `F`.\n\nWhen X and Y are seen as hexadecimal numbers, which is larger?\n\nConstraints\n\n* Each X and Y is `A`, `B`, `C`, `D`, `E` or `F`.\n\nInput\n\nInput is given from Standard Input in the following format:\n\n\nX Y\n\n\nOutput\n\nIf X is smaller, print `<`; if Y is smaller, print `>`; if they are equal, print `=`.\n\nExamples\n\nInput\n\nA B\n\n\nOutput\n\n<\n\n\nInput\n\nE C\n\n\nOutput\n\n>\n\n\nInput\n\nF F\n\n\nOutput\n\n=', 'role': 'user'}, {'content': 'Step 1:  We need to compare two hexadecimal numbers represented by the letters 

We now have to apply the chat template for `LFM` onto the conversations, and save it to `text`. We also remove the BOS token otherwise we'll get double BOS tokens!

In [7]:
def formatting_prompts_func(examples):
    texts = tokenizer.apply_chat_template(
        examples["conversations"],
        tokenize = False,
        add_generation_prompt = False,
    )
    return { "text" : [x.removeprefix(tokenizer.bos_token) for x in texts] }

dataset = dataset.map(formatting_prompts_func, batched = True)

In [8]:
dataset[37]["text"]

'<|im_start|>user\nWrite Python code to solve the task:\nIn programming, hexadecimal notation is often used.\n\nIn hexadecimal notation, besides the ten digits 0, 1, ..., 9, the six letters `A`, `B`, `C`, `D`, `E` and `F` are used to represent the values 10, 11, 12, 13, 14 and 15, respectively.\n\nIn this problem, you are given two letters X and Y. Each X and Y is `A`, `B`, `C`, `D`, `E` or `F`.\n\nWhen X and Y are seen as hexadecimal numbers, which is larger?\n\nConstraints\n\n* Each X and Y is `A`, `B`, `C`, `D`, `E` or `F`.\n\nInput\n\nInput is given from Standard Input in the following format:\n\n\nX Y\n\n\nOutput\n\nIf X is smaller, print `<`; if Y is smaller, print `>`; if they are equal, print `=`.\n\nExamples\n\nInput\n\nA B\n\n\nOutput\n\n<\n\n\nInput\n\nE C\n\n\nOutput\n\n>\n\n\nInput\n\nF F\n\n\nOutput\n\n=<|im_end|>\n<|im_start|>assistant\nStep 1:  We need to compare two hexadecimal numbers represented by the letters X and Y.\nStep 2:  We can define a list of hexadecimal le

In [9]:
# Create an evaluation set (5% of 1k rows = 50 rows for testing)
dataset = dataset.train_test_split(test_size=0.05)
train_data = dataset["train"]
eval_data = dataset["test"]

<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [10]:
# from trl import SFTTrainer, SFTConfig
# trainer = SFTTrainer(
#     model = model,
#     tokenizer = tokenizer,
#     train_dataset = dataset,
#     eval_dataset = None, # Can set up evaluation!
#     args = SFTConfig(
#         dataset_text_field = "text",
#         per_device_train_batch_size = 2,
#         gradient_accumulation_steps = 4, # Use GA to mimic batch size!
#         warmup_steps = 5,
#         # num_train_epochs = 1, # Set this for 1 full training run.
#         max_steps = 60,
#         learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
#         logging_steps = 1,
#         optim = "adamw_8bit",
#         weight_decay = 0.01,
#         lr_scheduler_type = "linear",
#         seed = 3407,
#         report_to = "none", # Use this for WandB etc
#     ),
# )

In [11]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,   # [UPDATED] Pass the 9.5k training split
    eval_dataset = eval_data,     # [UPDATED] Pass the 500 row evaluation split
    args = SFTConfig(
        dataset_text_field = "text",

        # --- HARDWARE SURVIVAL TWEAKS ---
        per_device_train_batch_size = 1,   # [CHANGED] Dropped to 1 to survive 16k context on 6GB VRAM
        gradient_accumulation_steps = 4,

        # --- TRAINING DURATION ---
        num_train_epochs = 1, # LETS DO A FULL RUN!
        # max_steps = 30,

        # --- LEARNING DYNAMICS ---
        learning_rate = 2e-4,
        warmup_steps = 5,
        lr_scheduler_type = "cosine",      # [CHANGED] Smoothly tapers the learning rate down, much better than "linear"
        optim = "adamw_8bit",
        weight_decay = 0.01,
        seed = 3407,

        # --- EVALUATION & LOGGING ---
        logging_steps = 1,                # [CHANGED] Log training loss at every step
        eval_strategy = "steps",     # [NEW] Tell it to evaluate during training
        eval_steps = 1,                   # [NEW] Check validation loss at every step
        report_to = "none"
    ),
)

Unsloth: Tokenizing ["text"]: 100%|████████████████████████████████████████████| 50/50 [00:00<00:00, 310.04 examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [12]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Filter: 100%|████████████████████████████████████████████████████████████████| 950/950 [00:01<00:00, 739.93 examples/s]


Unsloth: Removed 3 out of 950 samples from train_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


Filter: 100%|██████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 653.89 examples/s]


Let's verify masking the instruction part is done! Let's print the 100th row again.

In [13]:
tokenizer.decode(trainer.train_dataset[37]["input_ids"])

"<|startoftext|><|im_start|>user\nWhy does the north magnetic pole exhibit a more rapid variation in location compared to the south magnetic pole, as observed in historical data, such as changes from 1998 to 2015?<|im_end|>\n<|im_start|>assistant\nThe Earth's magnetic field is generated by a dynamo effect, which results from circulating electric currents in the planet's molten outer core. This dynamic process is not uniform, leading to differences in the behavior of the magnetic poles.\n\nThe unequal variation between the north and south magnetic poles can be attributed to the complex interplay of the Earth's internal dynamics. The convection of the molten iron in the outer core, driven by heat from the inner core, creates relative motion between the conducting fluid and the Earth's magnetic field. If this motion generates electric currents due to, for example, friction between different layers, it can sustain a magnetic dipole field similar to Earth's.\n\nDifferences in the rate of ch

Now let's print the masked out example - you should see only the answer is present:

In [14]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[37]["labels"]]).replace(tokenizer.pad_token, " ")

"                                               The Earth's magnetic field is generated by a dynamo effect, which results from circulating electric currents in the planet's molten outer core. This dynamic process is not uniform, leading to differences in the behavior of the magnetic poles.\n\nThe unequal variation between the north and south magnetic poles can be attributed to the complex interplay of the Earth's internal dynamics. The convection of the molten iron in the outer core, driven by heat from the inner core, creates relative motion between the conducting fluid and the Earth's magnetic field. If this motion generates electric currents due to, for example, friction between different layers, it can sustain a magnetic dipole field similar to Earth's.\n\nDifferences in the rate of change between the poles likely stem from variations in the geological topology and fluid dynamics within the Earth's core. Although the exact details are not fully understood, large-scale computer simu

In [17]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 3050 6GB Laptop GPU. Max memory = 6.0 GB.
6.379 GB of memory reserved.


In [18]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 947 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 11,108,352 of 1,181,448,960 (0.94% trained)


Step,Training Loss,Validation Loss
1,0.504431,0.936355
2,1.084257,0.931706
3,0.558703,0.932355
4,1.083121,0.913373
5,1.112410,0.889044
6,0.750632,0.859265
7,0.572898,0.843241
8,0.883111,0.828378
9,0.639687,0.816794
10,0.954632,0.811403


In [19]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

889.2915 seconds used for training.
14.82 minutes used for training.
Peak reserved memory = 8.404 GB.
Peak reserved memory for training = 2.025 GB.
Peak reserved memory % of max memory = 140.067 %.
Peak reserved memory for training % of max memory = 33.75 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!



In [20]:
messages = [{
    "role": "user",
    "content": "Write a poem about a sloth.",
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 128, # Increase for longer outputs!
    # Recommended Liquid settings!
    temperature = 0.1, top_k = 50, top_p = 0.1, repetition_penalty = 1.05,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

In the jungle, where the trees sway,
A sloth slips by, in a slow parade.
With a snout so wide, and eyes so bright,
He moves with a pace that's quite a sight.

His fur is soft, like a cloud in the sky,
And his tail is long, like a ropey stringy.
He slips through the leaves, with a gentle sway,
And disappears into the green, like a shadowy play.

He eats leaves, and fruits, and berries too,
And sleeps for days, in a cozy little loo.
But when he wakes up, with


In [21]:
messages = [{
    "role": "user",
    "content": "Find the minima and maxima of 3x^2 - 5x^2 + 6",
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 1024, # Increase for longer outputs!
    # Recommended Liquid settings!
    temperature = 0.5, top_k = 50, top_p = 0.1, repetition_penalty = 1.05,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

To find the minima and maxima of the function $f(x) = 3x^2 - 5x^2 + 6$, we need to find the critical points by taking the derivative of the function and setting it equal to zero.

First, let's simplify the function:
$f(x) = 3x^2 - 5x^2 + 6$
$f(x) = -2x^2 + 6$

Now, take the derivative of the function with respect to x:
$f'(x) = -4x$

Set the derivative equal to zero to find the critical points:
$-4x = 0$
$x = 0$

Now, we need to determine whether this critical point corresponds to a minimum or maximum. We can do this by examining the second derivative of the function:

$f''(x) = -4$

Since the second derivative is negative, the critical point at $x=0$ corresponds to a maximum.

To find the value of the function at the maximum, substitute $x=0$ into the original function:
$f(0) = 3(0)^2 - 5(0)^2 + 6$
$f(0) = 6$

Therefore, the function has a maximum value of 6 at $x=0$.

There are no other critical points, so there are no minima in this case.<|im_end|>


In [22]:
messages = [{
    "role": "user",
    "content": "Given a list of 3 distinct numbers like [0, 1, 1, 2, 0], return its sorted list without sorting, use 3 pointer approach. Use Python",
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 1024, # Increase for longer outputs!
    # Recommended Liquid settings!
    temperature = 0.5, top_k = 50, top_p = 0.1, repetition_penalty = 1.05,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

To solve this problem using the 3-pointer approach, we can follow these steps:

1. Initialize three pointers: `i`, `j`, and `k`.
   - `i` will be used to traverse the list from left to right.
   - `j` will be used to traverse the list from right to left.
   - `k` will be used to traverse the list from left to right.

2. Initialize a variable `max_val` to store the maximum value in the current window of size 3.
   - We can initialize it with the first element of the list.

3. Traverse the list from left to right using the `i` pointer.
   - For each element, update `max_val` if the current element is greater than `max_val`.

4. Move the `j` pointer to the right until the window size becomes 3.
   - If the window size becomes 3, update `max_val` by comparing it with the elements at indices `i`, `i+1`, and `i+2`.
   - If the element at index `i+2` is greater than `max_val`, update `max_val` to the element at index `i+2`.

5. Move the `k` pointer to the left until the window size becomes 3.

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [23]:
ROOT_PATH = "D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-16k-Instruct"
import os
os.makedirs(ROOT_PATH, exist_ok=True)

In [24]:
model.save_pretrained(f"{ROOT_PATH}/lora_r16_8bit_model")  # Local saving
tokenizer.save_pretrained(f"{ROOT_PATH}/lora_r16_8bit_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('D:/ML/Models/Finetuned/LiquidAI-LFM-1.2B-16k-Instruct/lora_r16_8bit_model\\tokenizer_config.json',
 'D:/ML/Models/Finetuned/LiquidAI-LFM-1.2B-16k-Instruct/lora_r16_8bit_model\\chat_template.jinja',
 'D:/ML/Models/Finetuned/LiquidAI-LFM-1.2B-16k-Instruct/lora_r16_8bit_model\\tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [25]:
if True:
    print("Let's load the saved LORA model")
    from unsloth import FastModel
    from transformers import Lfm2ForCausalLM
    print("Loading LORA r16 8 bit")
    model, tokenizer = FastModel.from_pretrained(
        model_name = f"{ROOT_PATH}/lora_r16_8bit_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 4096,
        # dtype = dtype,
        # auto_model = Lfm2ForCausalLM,
        load_in_4bit = True,
    )
    FastModel.for_inference(model) # Enable native 2x faster inference

messages = [{
    "role": "user",
    "content": "How do I code up a transformer?",
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 256, # Increase for longer outputs!
    # Recommended Liquid settings!
    temperature = 0.5, top_k = 50, top_p = 0.9, repetition_penalty = 1.05,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Let's load the saved LORA model
Loading LORA r16 8 bit
==((====))==  Unsloth 2026.3.4: Fast Lfm2 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 3050 6GB Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|█████| 148/148 [00:06<00:00, 24.55it/s, Materializing param=model.layers.15.operator_norm.weight]


To code up a transformer, you can follow these steps:

1. Define the input and output data structures: Determine the shape of the input and output data structures. For example, if you are working with images, you may need to define the input as a 2D array and the output as a 3D array.
2. Implement the encoder and decoder layers: The encoder layer is responsible for encoding the input data into a lower-dimensional representation. The decoder layer is responsible for decoding the output data back into the original input format. You can use convolutional layers or recurrent layers to implement these layers.
3. Add batch normalization and dropout layers: Batch normalization and dropout layers can be added to improve the performance of your transformer model. Batch normalization normalizes the input data to have zero mean and unit variance, while dropout randomly drops out a certain percentage of neurons during training to prevent overfitting.
4. Implement the attention mechanism: The atten

### Load f16 version

In [3]:
from unsloth import FastLanguageModel

# Load the ALREADY MERGED model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = f"{ROOT_PATH}/sft_with_unsloth",
    load_in_4bit = True, # FastLanguageModel needs this to initialize
)

==((====))==  Unsloth 2026.3.4: Fast Lfm2 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 3050 6GB Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|█████| 148/148 [00:03<00:00, 45.86it/s, Materializing param=model.layers.15.operator_norm.weight]


You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
# if False:
#     # I highly do NOT suggest - use Unsloth if possible
#     from peft import AutoPeftModelForCausalLM
#     from transformers import AutoTokenizer
#     model = AutoPeftModelForCausalLM.from_pretrained(
#         "lora_model", # YOUR MODEL YOU USED FOR TRAINING
#         load_in_4bit = load_in_4bit,
#     )
#     tokenizer = AutoTokenizer.from_pretrained("lora_model")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
#! NOT NEEDED HERE
# # Merge to 16bit
# if True: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
# if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# # Merge to 4bit
# if False: model.save_pretrained_merged("model_4bit", tokenizer, save_method = "merged_4bit_forced",)
# if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# # Just LoRA adapters
# if False:
#     model.save_pretrained("model")
#     tokenizer.save_pretrained("model")
# if False:
#     model.push_to_hub("hf/model", token = "")
#     tokenizer.push_to_hub("hf/model", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [26]:
# # Save to 8bit Q8_0
# if False: model.save_pretrained_gguf("model", tokenizer,)
# # Remember to go to https://huggingface.co/settings/tokens for a token!
# # And change hf to your username!
# if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# # Save to 16bit GGUF
# if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
# if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# # Save to q4_k_m GGUF
# if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
# if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
# if True:
#     # model.push_to_hub_gguf(
#       model.save_pretrained_gguf(
#         "models", # Change hf to your username!
#         tokenizer,
#         quantization_method = ["q4_k_m", "q8_0", "f16"],
#         token = "",
#     )

if True:
    model.save_pretrained_gguf(
        f"{ROOT_PATH}/sft_with_unsloth",
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "f16"], 
    )


Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: D:\Cache\hf\hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `D:/ML/Models/Finetuned/LiquidAI-LFM-1.2B-16k-Instruct/sft_with_unsloth`: 100%|█


Successfully copied all 1 files from cache to `D:/ML/Models/Finetuned/LiquidAI-LFM-1.2B-16k-Instruct/sft_with_unsloth`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|███████████████████████████████████████████████| 1/1 [00:07<00:00,  7.90s/it]


Unsloth: Merge process complete. Saved to `D:\ML\Models\Finetuned\LiquidAI-LFM-1.2B-16k-Instruct\sft_with_unsloth`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m', 'q8_0', 'f16'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['D:/ML/Models/Finetuned/LiquidAI-LFM-1.2B-16k-Instruct/sft_with_unsloth_gguf\\LFM2.5-1.2B-Instruct.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: [2] Converting GGUF bf16 into q8_0. This might take 10 minutes...
Unsloth: [2] Converting GGUF bf16 into f16. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['D:/ML/Models/Finetuned/LiquidAI-LFM-1.2B-16k-Instruct/sft_with_unsloth_gguf\\LFM2.5-1.2B-Instruct.F16.gguf', 'D:/ML/Models/Finetuned/LiquidAI-LFM-1.2B-16k-Instruct/sft_with_unsloth_gguf\\LFM2.5-1.2B-Instruct.Q8_0.gguf', 'D:/ML/Models/Finetuned/LiquidAI-LFM-1.2B-16k-Instruct/sft_with_unsloth_gguf\\LFM2.5-1.2B-Instruct.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'LiquidAI/LFM2.5-1.2B-Instruct'. Skipping Ol

### DONE!